In [10]:
import torch.nn as nn
import torch
import pandas as pd
import sklearn

In [11]:
data = pd.read_csv("Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


## Descripción de la tarea

Trabajaremos con un dataset de la serie "Pokémon" que desglosa las capacidades de cada criatura en atributos cuantitativos. La tarea consiste en utilizar arquitecturas de redes neuronales simples (MLP) para identificar patrones que distingan a los Pokémon Legendarios del resto. Deberán procesar estas variables y entrenar un clasificador que maximice la capacidad predictiva sobre la variable objetivo "Legendary".

Consideraciones:
- Debe entregarlo a más tardar el 29 de mayo a las 18:00 horas.
- Debe ser entregado al correo luis.llanca@uach.cl con el asunto "Tarea-1-MLP", el archivo debe llamarse NG-MLP-Tarea1.ipynb donde NG es el número de grupo. Es importante que el asunto sea exactamente el mismo. También, se les pedirá que se anoten en la plantilla (que se compartirá posteriormente) para una pequeña interrogación.
- Por cada 20 minutos de retraso, se descontará una décima de la nota. 
- Si necesitan ayuda, pueden escribir a los correos luis.llanca@uach.cl, luis.llanca@cenia.cl o escribir al discord de usuario: llanking (tengo una foto de mi gata). PD: prefiero mucho más el discord. 

### Explicación del dataset
Explique el dataset en detalle, incluyendo como mínimo una pequeña descripción de cada columna, el tipo de datos que contiene cada columna, y cualquier información adicional relevante para entender el dataset.
## '#'
Indica la id del pokemon en la tabla
## Name
nos muestra el nombre de este, este dato al no ser un dato numerico será remplazado por 27 nuevas columnas, la primera name_length, y las otras 26 son para las letras del abecedario a,b,c,d,e,f,g,h,i,j,k... con esto se espera que la red pueda aprender de la cantidad y letras de cada nombre de los pokemones y tambien del largo de su nombre, porque al revisar el dataset se vieron muchos pokemones Mega que tambien pueden tener altas estadisticas, pero todos sus nombres son muy largos. con este cambio en el data set se espera un mejor aprendizaje, 
## Type 1
Es una categoría igualmente alfabetica, pero esta vez se hizo algo similar con el metodo get_Dummies,  el cual creo otra lista de vectores con 1 y 0 para saber si un pokemon pertenece a una clase o a otra
## Type 2 
lo mismo que tipo 1
## Total 	
La suma total de las estadisticas pokemon, un buen dato para saber que tan fuerte es, se debe normalizar para que la red neuronal no le de solo importancia a esto
## HP
Vida del pokemon, las siguientes 5 columnas son todas estadisticas bases que seran valiosas para el aprendizaje, igualmente se nomalizaran y se espera que un pokemon legendario tenga alguna o varias de estas mas elevada que el resto. (entre el 90 y el 100 de una stat)
## Attack
## Defense
## Sp. Atk
## Sp. Def
## Speed

## Generation
Supongo que aqui aprende la antiguedad del pokemon? quizás sirva aunque suena un poko descartable, si reconoce un patron de cantidad de legendarios por generacion puede ser util
## Legendary
La estadistica a ser estudiada, son pokemones mas fuertes que los comunes, pero no pueden evolucionar como un pokemon normal.

### Preparación del dataset

Realice una preparación del dataset según lo que considere necesario para el entrenamiento de un modelo de clasificación. Justifique las decisiones que tome en este proceso.

In [55]:
import pandas as pd
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# 1. Cargar los datos
df = pd.read_csv('Pokemon.csv')
# Asegurarnos de que los nombres sean texto para que no tire error más adelante
df['Name'] = df['Name'].fillna('').astype(str)
# Vemos si tiene espacios (nombres compuestos como Tapu Koko)
df['Has_Space'] = df['Name'].apply(lambda x: 1 if ' ' in x else 0)
# Largo del nombre
df['Name_Length'] = df['Name'].apply(len)
# Vector con las letras del abecedario (A-Z)
# Esto ayuda a la red a notar si el nombre usa letras "raras" típicas de legendarios (Z, X, Y, Q)
for letra in string.ascii_lowercase:
    df[f'char_{letra}'] = df['Name'].apply(lambda x: 1 if letra in x.lower() else 0)
# Borrar el nombre original porque la red neuronal no procesa texto crudo
df = df.drop(['Name'], axis=1)
# Transformar los Tipos (Agua, Fuego, etc.) a columnas de 1s y 0s
# dummy_na=False evita problemas si un pokemon no tiene Tipo 2
df = pd.get_dummies(df, columns=['Type 1', 'Type 2'], dummy_na=False)
# Separar lo que queremos predecir (y) de los datos (X)
y = df['Legendary'].astype(int).values
# Borramos Legendary de X, y también la columna '#' (el número de la pokedex) si es que viene en el csv
X = df.drop(['Legendary', '#'], axis=1, errors='ignore')
# Convertimos todo a una matriz plana
X = X.values
# Escalar los datos numéricos (como HP, Ataque, Total)
# Esto ayuda a que el modelo entrene más rápido y no le dé más peso al stat 'Total' solo por ser un número más grande
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# 1. Primera división: 60% Entrenamiento, 40% Temporal (Validación + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.40, random_state=42)

# 2. Segunda división: Separar el 40% temporal en Validación (30% del total) y Test (10% del total)
# El 10% es la cuarta parte (0.25) del 40% temporal.
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
# Tensores con sklearn
# Tensores de Entrenamiento (60%)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

# Tensores de Validación (30%)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Tensores de Prueba/Test (10%)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Comprobación rápida para ver cuántos Pokémon quedaron en cada set
print(f"Total datos Entrenamiento: {len(X_train_tensor)} (60%)")
print(f"Total datos Validación: {len(X_val_tensor)} (30%)")
print(f"Total datos Test: {len(X_test_tensor)} (10%)")

# Guardamos la cantidad de columnas final para pasársela a los modelos
input_size_final = X_train_tensor.shape[1]

print("¡Datos preparados!")
print(f"Cantidad de neuronas de entrada necesarias: {input_size_final}")

Total datos Entrenamiento: 480 (60%)
Total datos Validación: 160 (30%)
Total datos Test: 160 (10%)
¡Datos preparados!
Cantidad de neuronas de entrada necesarias: 72


### Definición del modelo  
Defina al menos 3 arquitecturas de redes neuronales simples (MLP) para el problema de clasificación. Justifique las decisiones que tome en la definición de cada arquitectura. Las definiciones se deben hacer en un archivo ```models.py``` e importarlas en este cuadernillo. Debe seleccionar "la mejor" arquitectura para el entrenamiento, y justificar su elección.

In [56]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Importamos los modelos desde el archivo models.py
from models import SimpleMLP, MediumMLP

# 2. Instanciamos el modelo 1 
# (input_size_final es la variable que guardamos en la preparación de datos)
modelo_actual = SimpleMLP(input_size_final)

# 3. Definimos la pérdida y el optimizador
criterio = nn.BCEWithLogitsLoss()
# Le pasamos los parámetros del modelo al optimizador Adam
optimizador = optim.Adam(modelo_actual.parameters(), lr=0.001)

epocas = 100

# 4. El ciclo de entrenamiento
for epoca in range(epocas):
    
    # --- FASE DE ENTRENAMIENTO ---
    modelo_actual.train() # Le decimos a PyTorch que estamos entrenando
    
    # Pasada hacia adelante
    predicciones_train = modelo_actual(X_train_tensor)
    loss_train = criterio(predicciones_train, y_train_tensor)
    
    # Aprender de los errores (Backpropagation)
    optimizador.zero_grad() # Limpiamos los gradientes anteriores
    loss_train.backward()   # Calculamos dónde nos equivocamos
    optimizador.step()      # Ajustamos los pesos de la red
    
    # --- FASE DE VALIDACIÓN ---
    modelo_actual.eval() # Le decimos a PyTorch que ahora solo vamos a evaluar
    
    # torch.no_grad() apaga el cálculo de gradientes, 
    # ahorra memoria porque aquí no necesitamos que la red aprenda
    with torch.no_grad():
        predicciones_val = modelo_actual(X_val_tensor)
        loss_val = criterio(predicciones_val, y_val_tensor)
        # 1. Aplicar Sigmoide para transformar los números crudos a probabilidades de 0 a 1
        probabilidades = torch.sigmoid(predicciones_val)
        
        # 2. Si la probabilidad es mayor a 0.5, decimos que es Legendario (1), si no, normal (0)
        predicciones_finales = (probabilidades >= 0.5).float()
        
        # 3. Contar cuántos aciertos tuvimos comparando con las respuestas reales (y_val_tensor)
        aciertos = (predicciones_finales == y_val_tensor).sum().item()
        total_datos = y_val_tensor.size(0)
        
        # 4. Calcular el porcentaje
        accuracy_val = (aciertos / total_datos) * 100
        
    # Imprimimos cómo va todo cada 10 épocas
    if (epoca + 1) % 10 == 0:
        print(f"Época [{epoca+1}/{epocas}] | Loss Train: {loss_train.item():.4f} | Loss Val: {loss_val.item():.4f} | Acc Val: {accuracy_val:.2f}%")

Época [10/100] | Loss Train: 0.7041 | Loss Val: 0.7036 | Acc Val: 41.25%
Época [20/100] | Loss Train: 0.6193 | Loss Val: 0.6302 | Acc Val: 72.50%
Época [30/100] | Loss Train: 0.5467 | Loss Val: 0.5664 | Acc Val: 84.38%
Época [40/100] | Loss Train: 0.4832 | Loss Val: 0.5108 | Acc Val: 88.75%
Época [50/100] | Loss Train: 0.4275 | Loss Val: 0.4620 | Acc Val: 90.62%
Época [60/100] | Loss Train: 0.3787 | Loss Val: 0.4191 | Acc Val: 93.12%
Época [70/100] | Loss Train: 0.3362 | Loss Val: 0.3816 | Acc Val: 93.12%
Época [80/100] | Loss Train: 0.2991 | Loss Val: 0.3492 | Acc Val: 93.75%
Época [90/100] | Loss Train: 0.2669 | Loss Val: 0.3210 | Acc Val: 94.38%
Época [100/100] | Loss Train: 0.2388 | Loss Val: 0.2967 | Acc Val: 94.38%


In [57]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Importamos los modelos desde el archivo models.py
from models import SimpleMLP, MediumMLP

# 2. Instanciamos el modelo 1 
# (input_size_final es la variable que guardamos en la preparación de datos)
modelo_actual = MediumMLP(input_size_final)

# 3. Definimos la pérdida y el optimizador
criterio = nn.BCEWithLogitsLoss()
# Le pasamos los parámetros del modelo al optimizador Adam
optimizador = optim.Adam(modelo_actual.parameters(), lr=0.001)

epocas = 100

# 4. El ciclo de entrenamiento
for epoca in range(epocas):
    
    # --- FASE DE ENTRENAMIENTO ---
    modelo_actual.train() # Le decimos a PyTorch que estamos entrenando
    
    # Pasada hacia adelante
    predicciones_train = modelo_actual(X_train_tensor)
    loss_train = criterio(predicciones_train, y_train_tensor)
    
    # Aprender de los errores (Backpropagation)
    optimizador.zero_grad() # Limpiamos los gradientes anteriores
    loss_train.backward()   # Calculamos dónde nos equivocamos
    optimizador.step()      # Ajustamos los pesos de la red
    
    # --- FASE DE VALIDACIÓN ---
    modelo_actual.eval() # Le decimos a PyTorch que ahora solo vamos a evaluar
    
    # torch.no_grad() apaga el cálculo de gradientes, 
    # ahorra memoria porque aquí no necesitamos que la red aprenda
    with torch.no_grad():
        predicciones_val = modelo_actual(X_val_tensor)
        loss_val = criterio(predicciones_val, y_val_tensor)
        # 1. Aplicar Sigmoide para transformar los números crudos a probabilidades de 0 a 1
        probabilidades = torch.sigmoid(predicciones_val)
        
        # 2. Si la probabilidad es mayor a 0.5, decimos que es Legendario (1), si no, normal (0)
        predicciones_finales = (probabilidades >= 0.5).float()
        
        # 3. Contar cuántos aciertos tuvimos comparando con las respuestas reales (y_val_tensor)
        aciertos = (predicciones_finales == y_val_tensor).sum().item()
        total_datos = y_val_tensor.size(0)
        
        # 4. Calcular el porcentaje
        accuracy_val = (aciertos / total_datos) * 100
        
    # Imprimimos cómo va todo cada 10 épocas
    if (epoca + 1) % 10 == 0:
        print(f"Época [{epoca+1}/{epocas}] | Loss Train: {loss_train.item():.4f} | Loss Val: {loss_val.item():.4f} | Acc Val: {accuracy_val:.2f}%")

Época [10/100] | Loss Train: 0.5713 | Loss Val: 0.5621 | Acc Val: 91.25%
Época [20/100] | Loss Train: 0.4699 | Loss Val: 0.4630 | Acc Val: 91.25%
Época [30/100] | Loss Train: 0.3595 | Loss Val: 0.3606 | Acc Val: 91.25%
Época [40/100] | Loss Train: 0.2629 | Loss Val: 0.2746 | Acc Val: 91.25%
Época [50/100] | Loss Train: 0.1944 | Loss Val: 0.2157 | Acc Val: 91.25%
Época [60/100] | Loss Train: 0.1461 | Loss Val: 0.1766 | Acc Val: 91.25%
Época [70/100] | Loss Train: 0.1082 | Loss Val: 0.1480 | Acc Val: 94.38%
Época [80/100] | Loss Train: 0.0776 | Loss Val: 0.1267 | Acc Val: 96.25%
Época [90/100] | Loss Train: 0.0535 | Loss Val: 0.1117 | Acc Val: 96.25%
Época [100/100] | Loss Train: 0.0364 | Loss Val: 0.1010 | Acc Val: 96.25%


### Definición de optimizador y función de costo  
Defina un optimizador y una función de costo adecuado para el entrenamiento del modelo. Justifique sus decisiones.

### Entrenamiento del modelo
Entrene el modelo seleccionado utilizando el dataset preparado. Justifique las decisiones que tome en el proceso de entrenamiento, incluyendo la selección de hiperparámetros, el número de épocas, el tamaño del batch, etc.

### Evaluación del modelo
Evalúe el modelo utilizando métricas adecuadas para este problema de clasificación. Justifique la selección de las métricas utilizadas y discuta los resultados obtenidos. 

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?



### IA Generativa

Con el fin de ocupar IA Generativa de manera responsable, se les pide que respondan a las siguientes preguntas:
1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.
2. ¿En qué parte o partes de la tarea utilizó estas herramientas?